In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.decomposition import FastICA
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
merged_df = pd.read_csv("/content/drive/MyDrive/")

In [ ]:
merged_df.head(3)

In [ ]:
dates = pd.to_datetime(merged_df['Date'])
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

In [ ]:
merged_df = merged_df.drop(columns=['Fluctuations'])
merged_df

In [ ]:
target = merged_df['Close'].values

In [ ]:
cols = list(merged_df)[4:]
data = merged_df[cols].astype(float)

In [ ]:
def minmax_scaler(data):
    min_val = data.min(axis=0)
    max_val = data.max(axis=0)
    scaled_data = (data - min_val) / (max_val - min_val)
    return np.array(scaled_data), min_val, max_val

def minmax_inverse_transform(scaled_data, min_val, max_val):
    return scaled_data * (max_val - min_val) + min_val

In [ ]:
data_scaled, min_val, max_val = minmax_scaler(data)

In [ ]:
# split to train data and test data
n_train = int(0.8*data_scaled.shape[0])
train_data_scaled = data_scaled[0: n_train]
train_dates = dates[0: n_train]

test_data_scaled = data_scaled[n_train:]
test_dates = dates[n_train:]

In [ ]:
train_data_scaled.shape, test_data_scaled.shape

In [ ]:
def reformat_data_for_LSTM(train_data_scaled, test_data_scaled, seq_len, pred_days):
    trainX = []
    trainY = []
    testX = []
    testY = []
    n_train = len(train_data_scaled)

    for i in range(seq_len, n_train - pred_days + 1):
        trainX.append(train_data_scaled[i - seq_len:i, 0:train_data_scaled.shape[1]])
        trainY.append(train_data_scaled[i + pred_days - 1:i + pred_days, 0])

    for i in range(seq_len, len(test_data_scaled) - pred_days + 1):
        testX.append(test_data_scaled[i - seq_len:i, 0:test_data_scaled.shape[1]])
        testY.append(test_data_scaled[i + pred_days - 1:i + pred_days, 0])

    trainX, trainY = np.array(trainX), np.array(trainY)
    testX, testY = np.array(testX), np.array(testY)

    return trainX, trainY, testX, testY

trainX_3, trainY_3, testX_3, testY_3 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 3, 1)
trainX_4, trainY_4, testX_4, testY_4 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 4, 1)
trainX_5, trainY_5, testX_5, testY_5 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 5, 1)
# trainX_14, trainY_14, testX_14, testY_14 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 14, 1)
# trainX_30, trainY_30, testX_30, testY_30 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 30, 1)
# trainX_60, trainY_60, testX_60, testY_60 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 60, 1)
# trainX_120, trainY_120, testX_120, testY_120 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 120, 1)

In [ ]:
print(trainX_4.shape, testX_4.shape)

In [ ]:
print(trainY_4.shape, testY_4.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output

class TCNWithAttention(nn.Module):
    def __init__(self, input_size, output_size, num_layers, kernel_size, attention_size, dilation):
        super(TCNWithAttention, self).__init__()
        self.conv_layers = nn.ModuleList([nn.Conv1d(input_size, input_size, kernel_size, dilation=dilation**i) for i in range(num_layers)])
        self.fc = nn.Linear(input_size, output_size)
        self.attn = nn.Linear(input_size*4, attention_size)

    def forward(self, input):
        input = input.permute(0, 2, 1)
        for layer in self.conv_layers:
            input = torch.relu(layer(input))
        input_flattened = input.view(input.size(0), -1)
        attn_weights = torch.softmax(self.attn(input_flattened), dim=1)
        expanded_attention_weights = attn_weights.unsqueeze(1)
        expanded_attention_weights = expanded_attention_weights.expand(-1, seq_len, -1)
        context = torch.sum(expanded_attention_weights * input, dim=2)
        output = self.fc(context)
        return output


class CombinedModel(nn.Module):
    def __init__(self, lstm_input_size, tcn_input_size, hidden_size, output_size, dilation):
        super(CombinedModel, self).__init__()
        self.lstm_with_attention = LSTMWithAttention(lstm_input_size, hidden_size, output_size)
        self.tcn_with_attention = TCNWithAttention(tcn_input_size, output_size, num_layers=3, kernel_size=1, attention_size=4, dilation=2)
        self.fc = nn.Linear(output_size * 2, output_size)

    def forward(self, lstm_input, tcn_input):
        lstm_output = self.lstm_with_attention(lstm_input)
        tcn_output = self.tcn_with_attention(tcn_input)

        combined_output = torch.cat((lstm_output, tcn_output), dim=0)
        output = (lstm_output + tcn_output) / 2
        return output


lstm_input_size = 19
tcn_input_size = 19
hidden_size = 64
output_size = 1
seq_len = 19

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)

dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)
batch_size = 4
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
combined_model = CombinedModel(lstm_input_size, tcn_input_size, hidden_size, output_size, dilation=2)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)

    return mape_score

optimizer = optim.Adam(combined_model.parameters(), lr=0.001)


num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = combined_model(batch_inputs, batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(mape)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

test_dataset = TensorDataset(testX_4_tensor, testY_4_tensor)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_loss = 0.0
    total_samples = 0

    total_predicted_li = []
    for test_inputs, test_targets in test_loader:
        test_prediction = combined_model(test_inputs, test_inputs)
        total_predicted_li.append(test_prediction)

        test_loss = criterion(test_prediction, test_targets)
        test_mape = MAPE(test_prediction, test_targets)

        total_loss += test_loss.item() * test_inputs.size(0)
        total_mape += test_mape.item() * test_inputs.size(0)
        total_samples += test_inputs.size(0)

    avg_loss = total_loss / total_samples
    avg_mape = total_mape / total_samples

    print(f"Test Loss: {avg_loss:.4f}, Test MAPE: {avg_mape:.4f}")

In [ ]:
flattened_arrays = [tensor.flatten().numpy() for tensor in total_predicted_li]

result_array = np.concatenate(flattened_arrays)
result_array

In [ ]:
plt.figure(figsize=(18, 10))

plt.title('All features')
plt.plot(test_dates[4:], result_array, color='black', label='Actual Close Price')
plt.plot(test_dates[4:], testY_4, color='red', linestyle='--', label='Predicted Close Price by ATFN')

plt.legend()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output

lstm_input_size = 19
hidden_size = 64
output_size = 1
seq_len = 19

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)

dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)

batch_size = 4

train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

lstm_model = LSTMWithAttention(lstm_input_size, hidden_size, output_size)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)
    return mape_score

optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = lstm_model(batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}, MAPE: {mape.item()}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

test_dataset = TensorDataset(testX_4_tensor, testY_4_tensor)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_loss = 0.0
    total_samples = 0

    total_predicted_li = []
    for test_inputs, test_targets in test_loader:
        test_prediction = lstm_model(test_inputs)

        total_predicted_li.append(test_prediction)

        test_loss = criterion(test_prediction, test_targets)
        test_mape = MAPE(test_prediction, test_targets)

        total_loss += test_loss.item() * test_inputs.size(0)
        total_mape += test_mape.item() * test_inputs.size(0)
        total_samples += test_inputs.size(0)

    avg_loss = total_loss / total_samples
    avg_mape = total_mape / total_samples

    print(f"Test Loss: {avg_loss:.4f}, Test MAPE: {avg_mape:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class TCNWithAttention(nn.Module):
    def __init__(self, input_size, output_size, num_layers, kernel_size, attention_size, dilation):
        super(TCNWithAttention, self).__init__()
        self.conv_layers = nn.ModuleList([nn.Conv1d(input_size, input_size, kernel_size, dilation=dilation**i) for i in range(num_layers)])
        self.fc = nn.Linear(input_size, output_size)
        self.attn = nn.Linear(input_size*4, attention_size)

    def forward(self, input):
        input = input.permute(0, 2, 1)
        for layer in self.conv_layers:
            input = torch.relu(layer(input))
        input_flattened = input.view(input.size(0), -1)
        attn_weights = torch.softmax(self.attn(input_flattened), dim=1)
        expanded_attention_weights = attn_weights.unsqueeze(1)
        expanded_attention_weights = expanded_attention_weights.expand(-1, input.size(1), -1)
        context = torch.sum(expanded_attention_weights * input, dim=2)
        output = self.fc(context)
        return output


tcn_input_size = 19
output_size = 1
seq_len = 19

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)

dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)

batch_size = 4
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

tcn_model = TCNWithAttention(input_size=tcn_input_size, output_size=output_size, num_layers=3, kernel_size=1, attention_size=4, dilation=2)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)
    return mape_score

optimizer = optim.Adam(tcn_model.parameters(), lr=0.001)

num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = tcn_model(batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}, MAPE: {mape.item()}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

test_dataset = TensorDataset(testX_4_tensor, testY_4_tensor)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_loss = 0.0
    total_samples = 0

    total_predicted_li = []
    for test_inputs, test_targets in test_loader:
        test_prediction = tcn_model(test_inputs)

        total_predicted_li.append(test_prediction)

        test_loss = criterion(test_prediction, test_targets)
        test_mape = MAPE(test_prediction, test_targets)

        total_loss += test_loss.item() * test_inputs.size(0)
        total_mape += test_mape.item() * test_inputs.size(0)
        total_samples += test_inputs.size(0)

    avg_loss = total_loss / total_samples
    avg_mape = total_mape / total_samples

    print(f"Test Loss: {avg_loss:.4f}, Test MAPE: {avg_mape:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMWithoutAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMWithoutAttention, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        last_hidden_state = lstm_out[:, -1, :]
        output = self.fc(last_hidden_state)
        return output


lstm_input_size = 19
hidden_size = 64
output_size = 1
seq_len = 19

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)

dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)

batch_size = 4
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

lstm_model = LSTMWithoutAttention(input_size=lstm_input_size, hidden_size=hidden_size, output_size=output_size)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)
    return mape_score

optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = lstm_model(batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}, MAPE: {mape.item()}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

test_dataset = TensorDataset(testX_4_tensor, testY_4_tensor)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_loss = 0.0
    total_samples = 0

    total_predicted_li = []
    for test_inputs, test_targets in test_loader:
        test_prediction = lstm_model(test_inputs)

        total_predicted_li.append(test_prediction)

        test_loss = criterion(test_prediction, test_targets)
        test_mape = MAPE(test_prediction, test_targets)

        total_loss += test_loss.item() * test_inputs.size(0)
        total_mape += test_mape.item() * test_inputs.size(0)
        total_samples += test_inputs.size(0)

    avg_loss = total_loss / total_samples
    avg_mape = total_mape / total_samples

    print(f"Test Loss: {avg_loss:.4f}, Test MAPE: {avg_mape:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class TCNWithoutAttention(nn.Module):
    def __init__(self, input_size, output_size, num_layers, kernel_size, dilation):
        super(TCNWithoutAttention, self).__init__()
        self.conv_layers = nn.ModuleList([nn.Conv1d(input_size, input_size, kernel_size, dilation=dilation**i) for i in range(num_layers)])
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, input):
        input = input.permute(0, 2, 1)

        for layer in self.conv_layers:
            input = torch.relu(layer(input))

        input_flattened = input.view(input.size(0), -1)

        self.fc = nn.Linear(76, output_size)
        output = self.fc(input_flattened)
        return output


tcn_input_size = 19
output_size = 1
seq_len = 19
num_layers = 3
kernel_size = 1
dilation = 3

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)

dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)

batch_size = 4

train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

tcn_model_without = TCNWithoutAttention(input_size=tcn_input_size, output_size=output_size, num_layers=num_layers, kernel_size=kernel_size, dilation=dilation)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)
    return mape_score

optimizer = optim.Adam(tcn_model_without.parameters(), lr=0.001)

num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = tcn_model_without(batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}, MAPE: {mape.item()}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

test_dataset = TensorDataset(testX_4_tensor, testY_4_tensor)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_loss = 0.0
    total_samples = 0

    total_predicted_li = []

    for test_inputs, test_targets in test_loader:
        test_prediction = tcn_model_without(test_inputs)

        total_predicted_li.append(test_prediction)

        test_loss = criterion(test_prediction, test_targets)

        test_mape = MAPE(test_prediction, test_targets)

        total_loss += test_loss.item() * test_inputs.size(0)
        total_mape += test_mape.item() * test_inputs.size(0)
        total_samples += test_inputs.size(0)

    avg_loss = total_loss / total_samples
    avg_mape = total_mape / total_samples

    print(f"Test Loss: {avg_loss:.4f}, Test MAPE: {avg_mape:.4f}")

    all_predictions = torch.cat(total_predicted_li, dim=0).cpu().numpy()
    all_targets = testY_4_tensor.cpu().numpy()
